In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("f1-spark").getOrCreate()

In [2]:
laps = spark.read.parquet("/home/jovyan/work/data/raw/2024/bahrain/laps.parquet")
laps.printSchema()
laps.count() 

root
 |-- Time: double (nullable = true)
 |-- Driver: string (nullable = true)
 |-- DriverNumber: string (nullable = true)
 |-- LapTime: double (nullable = true)
 |-- LapNumber: double (nullable = true)
 |-- Stint: double (nullable = true)
 |-- PitOutTime: double (nullable = true)
 |-- PitInTime: double (nullable = true)
 |-- Sector1Time: double (nullable = true)
 |-- Sector2Time: double (nullable = true)
 |-- Sector3Time: double (nullable = true)
 |-- Sector1SessionTime: double (nullable = true)
 |-- Sector2SessionTime: double (nullable = true)
 |-- Sector3SessionTime: double (nullable = true)
 |-- SpeedI1: double (nullable = true)
 |-- SpeedI2: double (nullable = true)
 |-- SpeedFL: double (nullable = true)
 |-- SpeedST: double (nullable = true)
 |-- IsPersonalBest: boolean (nullable = true)
 |-- Compound: string (nullable = true)
 |-- TyreLife: double (nullable = true)
 |-- FreshTyre: boolean (nullable = true)
 |-- Team: string (nullable = true)
 |-- LapStartTime: double (nullable =

1129

In [3]:
results = spark.read.parquet("/home/jovyan/work/data/raw/2024/bahrain/results.parquet")
results.printSchema()
results.count() 

root
 |-- DriverNumber: string (nullable = true)
 |-- BroadcastName: string (nullable = true)
 |-- Abbreviation: string (nullable = true)
 |-- DriverId: string (nullable = true)
 |-- TeamName: string (nullable = true)
 |-- TeamColor: string (nullable = true)
 |-- TeamId: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- FullName: string (nullable = true)
 |-- HeadshotUrl: string (nullable = true)
 |-- CountryCode: string (nullable = true)
 |-- Position: double (nullable = true)
 |-- ClassifiedPosition: string (nullable = true)
 |-- GridPosition: double (nullable = true)
 |-- Q1: double (nullable = true)
 |-- Q2: double (nullable = true)
 |-- Q3: double (nullable = true)
 |-- Time: double (nullable = true)
 |-- Status: string (nullable = true)
 |-- Points: double (nullable = true)
 |-- Laps: double (nullable = true)



20

In [4]:
telemetry = spark.read.parquet("/home/jovyan/work/data/raw/2024/bahrain/telemetry.parquet")
telemetry.printSchema()
telemetry.count() 

root
 |-- Date: timestamp_ntz (nullable = true)
 |-- RPM: double (nullable = true)
 |-- Speed: double (nullable = true)
 |-- nGear: long (nullable = true)
 |-- Throttle: double (nullable = true)
 |-- Brake: boolean (nullable = true)
 |-- DRS: long (nullable = true)
 |-- Source: string (nullable = true)
 |-- Time: double (nullable = true)
 |-- SessionTime: double (nullable = true)
 |-- Driver: string (nullable = true)



696380

In [9]:
(laps
    .filter((F.col("IsAccurate") == True) & (F.col("LapTime").isNotNull()))
    .select("Driver", "LapNumber", "LapTime", "Compound", "TyreLife")
    .orderBy(F.col("LapTime").asc())
    .show(10)
)

+------+---------+-------+--------+--------+
|Driver|LapNumber|LapTime|Compound|TyreLife|
+------+---------+-------+--------+--------+
|   VER|     39.0| 92.608|    SOFT|     2.0|
|   LEC|     36.0|  94.09|    HARD|     2.0|
|   VER|     43.0| 94.151|    SOFT|     6.0|
|   VER|     46.0| 94.193|    SOFT|     9.0|
|   ALO|     48.0| 94.199|    HARD|     7.0|
|   VER|     47.0| 94.239|    SOFT|    10.0|
|   VER|     45.0| 94.249|    SOFT|     8.0|
|   VER|     51.0|  94.34|    SOFT|    14.0|
|   VER|     48.0|  94.35|    SOFT|    11.0|
|   PER|     40.0| 94.364|    SOFT|     4.0|
+------+---------+-------+--------+--------+
only showing top 10 rows


In [11]:
telemetry.select("Driver").distinct().count()

20

In [13]:
rapidas = (laps
    .filter((F.col("IsAccurate") == True) & (F.col("LapTime").isNotNull()))
    .select("Driver", "LapNumber", "LapTime", "Compound", "TyreLife")
    .orderBy(F.col("LapTime").asc())
)

In [14]:
rapidas.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [LapTime#3 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(LapTime#3 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=323]
      +- Project [Driver#1, LapNumber#4, LapTime#3, Compound#19, TyreLife#20]
         +- Filter ((isnotnull(IsAccurate#30) AND IsAccurate#30) AND isnotnull(LapTime#3))
            +- FileScan parquet [Driver#1,LapTime#3,LapNumber#4,Compound#19,TyreLife#20,IsAccurate#30] Batched: true, DataFilters: [isnotnull(IsAccurate#30), IsAccurate#30, isnotnull(LapTime#3)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/data/raw/2024/bahrain/laps.parquet], PartitionFilters: [], PushedFilters: [IsNotNull(IsAccurate), EqualTo(IsAccurate,true), IsNotNull(LapTime)], ReadSchema: struct<Driver:string,LapTime:double,LapNumber:double,Compound:string,TyreLife:double,IsAccurate:b...




In [15]:
rapidas.show(10)

+------+---------+-------+--------+--------+
|Driver|LapNumber|LapTime|Compound|TyreLife|
+------+---------+-------+--------+--------+
|   VER|     39.0| 92.608|    SOFT|     2.0|
|   LEC|     36.0|  94.09|    HARD|     2.0|
|   VER|     43.0| 94.151|    SOFT|     6.0|
|   VER|     46.0| 94.193|    SOFT|     9.0|
|   ALO|     48.0| 94.199|    HARD|     7.0|
|   VER|     47.0| 94.239|    SOFT|    10.0|
|   VER|     45.0| 94.249|    SOFT|     8.0|
|   VER|     51.0|  94.34|    SOFT|    14.0|
|   VER|     48.0|  94.35|    SOFT|    11.0|
|   PER|     40.0| 94.364|    SOFT|     4.0|
+------+---------+-------+--------+--------+
only showing top 10 rows
